In [ ]:
!pip install -q transformers datasets sentencepiece accelerate evaluate sacrebleu

In [ ]:
import os
import torch
from datasets import load_dataset, Dataset
from transformers import (
    MT5ForConditionalGeneration,
    AutoTokenizer, # Changed from MT5Tokenizer
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)
import evaluate

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
base_dir = "/content/drive/MyDrive/shared/phinc/mt5"

In [ ]:
dataset = load_dataset("LingoIITGN/PHINC")

# Train-test split
dataset = dataset["train"].train_test_split(test_size=0.1)

train_data = dataset["train"]
val_data = dataset["test"]

In [ ]:

model_name = "google/mt5-small"  # you can use base later

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = MT5ForConditionalGeneration.from_pretrained(model_name)

# IMPORTANT MEMORY FIXES
# Disabled gradient checkpointing due to CheckpointError
# model.gradient_checkpointing_enable()
model.config.use_cache = False   # VERY IMPORTANT

In [ ]:
print(train_data.column_names)
train_data.select(range(5)).to_pandas() # Print first 5 rows with their actual data

In [ ]:
# def preprocess(example):
#     # Define a prefix for the input sequence to guide the model
#     prefix = "translate Hinglish to English: "

#     # Prepare the input sentence and the target translation
#     inputs = prefix + example["Sentence"]
#     targets = example["English_Translation"]

#     # Tokenize the input sentences
#     model_inputs = tokenizer(
#         inputs,
#         max_length=128,
#         truncation=True,
#         padding="max_length"
#     )

#     # Tokenize the target translations to create labels for training
#     labels = tokenizer(
#         targets,
#         max_length=128,
#         truncation=True,
#         padding="max_length"
#     )

#     # Assign the tokenized labels to the model_inputs dictionary
#     model_inputs["labels"] = labels["input_ids"]
#     return model_inputs


def preprocess(example):
    prefix = "translate Hinglish to English: "

    inputs = prefix + example["Sentence"]
    targets = example["English_Translation"]

    model_inputs = tokenizer(
        inputs,
        max_length=64,        # reduce from 128
        truncation=True
    )

    labels = tokenizer(
        targets,
        max_length=64,
        truncation=True
    )

    # Replace padding token id with -100
    labels_ids = labels["input_ids"]
    labels_ids = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in [labels_ids]
    ][0]

    model_inputs["labels"] = labels_ids

    return model_inputs

In [ ]:
# # Apply the preprocess function to tokenize the training and validation datasets
# # The remove_columns argument is used to remove the original 'Sentence' and 'English_Translation' columns
# # as they are no longer needed after tokenization, saving memory.

# train_data = train_data.map(preprocess, remove_columns=train_data.column_names)
# val_data = val_data.map(preprocess, remove_columns=val_data.column_names)

### Saving Tokenized Data

After tokenization, it's a good practice to save the processed dataset to disk, especially for large datasets, to avoid re-running the tokenization step every time. We will save them in subdirectories within the `base_dir`.

In [ ]:
# # Define paths to save tokenized data
# tokenized_train_path = os.path.join(base_dir, "tokenized_train_data")
# tokenized_val_path = os.path.join(base_dir, "tokenized_val_data")

# # Save the tokenized training data
# train_data.save_to_disk(tokenized_train_path)
# print(f"Tokenized training data saved to: {tokenized_train_path}")

# # Save the tokenized validation data
# val_data.save_to_disk(tokenized_val_path)
# print(f"Tokenized validation data saved to: {tokenized_val_path}")

### Loading Tokenized Data

If you have already tokenized and saved your data, you can load it directly from disk using `load_from_disk` without needing to re-tokenize. This is useful for continuing work in a new session or for sharing pre-processed data.

In [ ]:
from datasets import load_from_disk

# Define paths where tokenized data was saved
tokenized_train_path = os.path.join(base_dir, "tokenized_train_data")
tokenized_val_path = os.path.join(base_dir, "tokenized_val_data")

# Load the tokenized training data
loaded_train_data = load_from_disk(tokenized_train_path)
print(f"Tokenized training data loaded from: {tokenized_train_path}")

# Load the tokenized validation data
loaded_val_data = load_from_disk(tokenized_val_path)
print(f"Tokenized validation data loaded from: {tokenized_val_path}")

# Assign loaded data to train_data and val_data for the trainer
train_data = loaded_train_data
val_data = loaded_val_data

# You can verify by printing a sample
print("Sample of loaded training data:")
print(loaded_train_data[0])

In [ ]:
print(train_data[0])

In [ ]:
sample = train_data[0]["labels"]
print(sample)
print("Valid tokens:", [x for x in sample if x != -100])

In [ ]:
# data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    padding=True   # dynamic padding (IMPORTANT)
)

In [ ]:
bleu = evaluate.load("sacrebleu")
chrf = evaluate.load("chrf")

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = [[l for l in label if l != -100] for label in labels]
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    bleu_score = bleu.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
    chrf_score = chrf.compute(predictions=decoded_preds, references=decoded_labels)

    return {
        "bleu": bleu_score["score"],
        "chrf": chrf_score["score"]
    }

In [ ]:
import os

OUTPUT_DIR = os.path.join(base_dir, "checkpoints")

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    num_train_epochs=3,
    logging_steps=1,
    eval_strategy="no",
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    predict_with_generate=False,
    fp16=True,
    max_grad_norm=1.0,
    dataloader_pin_memory=True,
    dataloader_num_workers=2,
    report_to="none"
)

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
print(train_data[0])

In [ ]:
print("--- Debugging Data Samples ---")
sample = train_data[0]

# Decode input_ids
input_text = tokenizer.decode(sample['input_ids'], skip_special_tokens=False)
print(f"Decoded Input: {input_text}")

# Decode labels (ignoring -100 which is the ignore index for loss calculation)
label_ids = [l if l != -100 else tokenizer.pad_token_id for l in sample['labels']]
target_text = tokenizer.decode(label_ids, skip_special_tokens=False)
print(f"Decoded Target: {target_text}")

# Check if they are identical
if input_text.replace("translate Hinglish to English: ", "") == target_text:
    print("\nWARNING: Input and Target are identical! This is why loss is 0.")

In [ ]:
model.train()
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)
trainer.train()

In [ ]:
# trainer.train(resume_from_checkpoint=True)